# Module 1.4 — Assemble nano-SLM, Train & Generate
**DeskMate SLM Curriculum · Phase 1 · Notebook 05**

Run on **Google Colab** (free T4 recommended — ~8 min) or CPU (~40 min).

By the end you will have:
- A complete GPT-style nano-SLM (~550k parameters) with weight tying
- 10,000-step training run with cosine LR schedule and gradient clipping
- Three generated samples at different temperature/top-k settings
- A saved, reloadable checkpoint

Read `1.4_nano_slm.md` first for the full theory.

---


## Step 0 — Install & Seed

In [ ]:
%%capture
!pip install -q torch==2.3.1 matplotlib==3.9.0


In [ ]:
import random, math, pathlib, urllib.request, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
if device == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Step 1 — Runtime, Paths & Corpus

In [ ]:
try:
    import google.colab; RUNTIME = "colab"
    PROJECT_ROOT = pathlib.Path("/content/slm")
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = "kaggle"
        PROJECT_ROOT = pathlib.Path("/kaggle/working/slm")
    except ImportError:
        RUNTIME = "local"
        PROJECT_ROOT = pathlib.Path(".").resolve()

DATA_DIR   = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Runtime : {RUNTIME}")


In [ ]:
corpus_path = DATA_DIR / "tiny_shakespeare.txt"
if not corpus_path.exists():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, corpus_path)

text = corpus_path.read_text(encoding="utf-8")
chars  = sorted(set(text))
V      = len(chars)
stoi   = {c: i for i, c in enumerate(chars)}
itos   = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"Corpus {len(text):,} chars  Vocab {V}  Train {len(train_data):,}  Val {len(val_data):,}")


## Step 2 — Hyperparameters

In [ ]:
# Model config
D_MODEL    = 128
N_HEADS    = 4
N_LAYERS   = 6
BLOCK_SIZE = 128
DROPOUT    = 0.2

# Training config
BATCH_SIZE    = 32
MAX_STEPS     = 10_000
LR_MAX        = 3e-4
LR_MIN        = 3e-5     # cosine schedule floor
WARMUP_STEPS  = 200
EVAL_INTERVAL = 500
GRAD_CLIP     = 1.0      # gradient clipping max norm

print(f"d_model={D_MODEL}, n_heads={N_HEADS}, n_layers={N_LAYERS}, block_size={BLOCK_SIZE}")


In [ ]:
def get_batch(split):
    src = train_data if split == "train" else val_data
    ix  = torch.randint(len(src) - BLOCK_SIZE, (BATCH_SIZE,))
    x   = torch.stack([src[i            : i+BLOCK_SIZE  ] for i in ix])
    y   = torch.stack([src[i+1          : i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)


## Step 3 — Model Assembly

In [ ]:
class Head(nn.Module):
    # Single causal self-attention head.
    def __init__(self, d_model, d_head, block_size, dropout):
        super().__init__()
        self.d_head = d_head
        self.q = nn.Linear(d_model, d_head, bias=False)
        self.k = nn.Linear(d_model, d_head, bias=False)
        self.v = nn.Linear(d_model, d_head, bias=False)
        self.drop = nn.Dropout(dropout)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, _ = x.shape
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)
        scores  = q @ k.transpose(-2, -1) * self.d_head**-0.5
        scores  = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        weights = self.drop(scores.softmax(dim=-1))
        return weights @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        d_head = d_model // n_heads
        self.heads = nn.ModuleList([Head(d_model, d_head, block_size, dropout) for _ in range(n_heads)])
        self.proj  = nn.Linear(d_model, d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))


class FeedForward(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, block_size, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = FeedForward(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


In [ ]:
class NanoSLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb    = nn.Embedding(vocab_size, d_model)
        self.pos_emb    = nn.Embedding(block_size, d_model)
        self.drop_emb   = nn.Dropout(dropout)
        self.blocks     = nn.Sequential(*[
            Block(d_model, n_heads, block_size, dropout) for _ in range(n_layers)
        ])
        self.ln_final   = nn.LayerNorm(d_model)
        self.lm_head    = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: token embedding and LM head share the same matrix
        self.lm_head.weight = self.tok_emb.weight

        # Initialise weights (GPT-2 style)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        assert T <= self.block_size
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x   = self.drop_emb(tok + pos)
        x   = self.blocks(x)
        x   = self.ln_final(x)
        return self.lm_head(x)                      # (B, T, vocab_size)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits   = self(idx_cond)[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs  = logits.softmax(dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx    = torch.cat([idx, next_id], dim=1)
        self.train()
        return idx


In [ ]:
model = NanoSLM(V, D_MODEL, N_HEADS, N_LAYERS, BLOCK_SIZE, DROPOUT).to(device)

# Parameter breakdown
n_embed  = sum(p.numel() for p in model.tok_emb.parameters())
n_pos    = sum(p.numel() for p in model.pos_emb.parameters())
n_blocks = sum(p.numel() for p in model.blocks.parameters())
n_ln     = sum(p.numel() for p in model.ln_final.parameters())
# lm_head shares weights with tok_emb — don't double-count
n_total  = sum(p.numel() for p in model.parameters()) - n_embed  # subtract shared weight

print("Parameter breakdown:")
print(f"  tok_emb (shared with lm_head) : {n_embed:>8,}")
print(f"  pos_emb                        : {n_pos:>8,}")
print(f"  {N_LAYERS} blocks                      : {n_blocks:>8,}")
print(f"  final LayerNorm                : {n_ln:>8,}")
print(f"  ─────────────────────────────────────────")
print(f"  Total (excl. shared)           : {n_total+n_embed:>8,}  ({(n_total+n_embed)/1e6:.3f}M)")


## Step 4 — Cosine LR Schedule & Gradient Clipping

A flat LR of 3e-4 for 10k steps works but a cosine decay (with short warmup) is
more standard and usually reaches lower final loss. Gradient clipping (max norm 1.0)
prevents the occasional large gradient from destabilising training.


In [ ]:
def get_lr(step):
    # Linear warmup then cosine decay to LR_MIN
    if step < WARMUP_STEPS:
        return LR_MAX * step / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    cosine   = 0.5 * (1 + math.cos(math.pi * progress))
    return LR_MIN + (LR_MAX - LR_MIN) * cosine

# Preview the schedule
sample_steps = list(range(0, MAX_STEPS + 1, 500))
sample_lrs   = [get_lr(s) for s in sample_steps]
plt.figure(figsize=(8, 3))
plt.plot(sample_steps, sample_lrs, color="#4C72B0")
plt.xlabel("Step"); plt.ylabel("LR"); plt.title("Cosine LR Schedule with Warmup")
plt.tight_layout(); plt.show()


## Step 5 — Training Loop (10 000 Steps)

In [ ]:
@torch.no_grad()
def estimate_loss(eval_iters=200):
    model.eval()
    results = {}
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            logits = model(xb)
            losses[k] = F.cross_entropy(logits.view(-1, V), yb.view(-1))
        results[split] = losses.mean().item()
    model.train()
    return results

opt = torch.optim.AdamW(model.parameters(), lr=LR_MAX, betas=(0.9, 0.95), weight_decay=0.1)
train_losses, val_losses, steps_logged = [], [], []

torch.manual_seed(SEED)
init = estimate_loss()
print(f"Initial  train: {init['train']:.4f}  val: {init['val']:.4f}")


In [ ]:
for step in range(MAX_STEPS):
    # Update LR
    lr = get_lr(step)
    for pg in opt.param_groups:
        pg["lr"] = lr

    xb, yb = get_batch("train")
    logits  = model(xb)
    loss    = F.cross_entropy(logits.view(-1, V), yb.view(-1))

    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    opt.step()

    if step % EVAL_INTERVAL == 0 or step == MAX_STEPS - 1:
        est = estimate_loss()
        train_losses.append(est["train"])
        val_losses.append(est["val"])
        steps_logged.append(step)
        print(f"step {step:5d}  lr {lr:.2e}  train: {est['train']:.4f}  val: {est['val']:.4f}")

print("\nTraining complete.")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps_logged, train_losses, label="train loss", color="#4C72B0")
ax.plot(steps_logged, val_losses,   label="val loss",   color="#DD8452", linestyle="--")
ax.set_xlabel("Step"); ax.set_ylabel("Cross-entropy loss")
ax.set_title(f"nano-SLM ({(n_total+n_embed)/1e3:.0f}k params) — 10k Steps")
ax.legend(); plt.tight_layout()
plt.savefig(str(MODELS_DIR / "nano_slm_loss.png"), bbox_inches="tight")
plt.show()
print(f"Final  train: {train_losses[-1]:.4f}  val: {val_losses[-1]:.4f}")


## Step 6 — Generation Suite

Compare three sampling strategies. The same model, same starting token, different
temperature and top-k settings produce noticeably different output styles.


In [ ]:
def gen(prompt="\n", max_new=500, temperature=1.0, top_k=None):
    torch.manual_seed(SEED)
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    out = model.generate(ids, max_new, temperature=temperature, top_k=top_k)
    return decode(out[0].tolist())

# Grid of strategies
configs = [
    ("temperature=1.0  (default)",           dict(temperature=1.0,  top_k=None)),
    ("temperature=0.7, top_k=20  (focused)", dict(temperature=0.7,  top_k=20)),
    ("temperature=0.5, top_k=10  (precise)", dict(temperature=0.5,  top_k=10)),
]

for label, kwargs in configs:
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(gen(**kwargs)[:400])


In [ ]:
print("Observations:")
print("  temperature=1.0 : most varied, may wander or repeat")
print("  temperature=0.7 : reads most natural, good for most tasks")
print("  temperature=0.5 : very focused, can become repetitive")
print()
print("For DeskMate reply generation: temperature ~0.7, top_k 20-50")
print("is the typical starting point — professional tone, not robotic.")


## Step 7 — Save Checkpoint & Verify Reload

In [ ]:
cfg = {
    "vocab_size": V,
    "d_model":    D_MODEL,
    "n_heads":    N_HEADS,
    "n_layers":   N_LAYERS,
    "block_size": BLOCK_SIZE,
    "dropout":    DROPOUT,
    "chars":      chars,          # save the vocab so we can reload without the text file
}

ckpt_path = MODELS_DIR / "nano_slm_ckpt.pt"
torch.save({"config": cfg, "model_state": model.state_dict()}, ckpt_path)
print(f"Saved: {ckpt_path}  ({ckpt_path.stat().st_size / 1024:.1f} KB)")


In [ ]:
# Reload from scratch and verify output matches
ckpt = torch.load(ckpt_path, map_location=device)
c    = ckpt["config"]

reload_chars  = c["chars"]
reload_stoi   = {ch: i for i, ch in enumerate(reload_chars)}
reload_itos   = {i: ch for i, ch in enumerate(reload_chars)}
reload_encode = lambda s: [reload_stoi[ch] for ch in s]
reload_decode = lambda ids: "".join(reload_itos[i] for i in ids)

reloaded = NanoSLM(
    c["vocab_size"], c["d_model"], c["n_heads"],
    c["n_layers"], c["block_size"], c["dropout"]
).to(device)
reloaded.load_state_dict(ckpt["model_state"])
reloaded.eval()

# Generate from reloaded model and compare
torch.manual_seed(SEED)
ids_r  = torch.tensor([reload_encode("\n")], dtype=torch.long, device=device)
text_r = reload_decode(reloaded.generate(ids_r, 200, temperature=0.7, top_k=20)[0].tolist())

torch.manual_seed(SEED)
ids_o  = torch.tensor([encode("\n")], dtype=torch.long, device=device)
text_o = decode(model.generate(ids_o, 200, temperature=0.7, top_k=20)[0].tolist())

print("Original  :", text_o[:100])
print("Reloaded  :", text_r[:100])
print()
print(f"Outputs match: {text_o == text_r}")


## Checkpoint

> **Explain the full path from a prompt string to the next generated token.**

Trace every step: tokenize → tensor → crop → token embed + pos embed → dropout →
N blocks (pre-norm MHA with causal mask + residual; pre-norm FF + residual) →
final LayerNorm → LM head → last position logits → temperature → top-k → softmax →
multinomial sample → decode.


In [ ]:
answer = """
[Write your answer here — trace all steps]
"""
print(answer)


## Deliverable Summary

| Artifact | Location |
|---|---|
| Loss curve | `models/nano_slm_loss.png` |
| Trained checkpoint | `models/nano_slm_ckpt.pt` |

**What you've built:** a complete GPT-style language model trained from scratch.
Every piece — tokenizer, embeddings, attention, MLP, residuals, generation — you wrote yourself.

**Next:** Module 1.5 — Why we stop here. You'll look at scaling law numbers and understand
why a *capable* domain SLM from scratch is infeasible on free tier, making the case for
the pretrained-base + fine-tuning approach that drives Phases 2–9.
